# Aegis - Semantic-Attack Detection (PAIR): L1 vs L2

Obfuscation attacks change the SURFACE (Base64, character-spacing) - L0/L1 handle those. **Semantic** attacks (PAIR/TAP/PAP) keep the surface fluent and benign-looking while preserving harmful intent, so a surface/pattern detector (L1) has little to grab. This scores real **PAIR** jailbreaks (JailbreakBench artifacts) with the L1 detectors vs an L2 content guard (Qwen3Guard), with benign over-refusal as the control.

**Setup:** Accelerator = GPU T4 (for the guards); Internet ON; Kaggle Secrets `HF_TOKEN` + `WANDB_API_KEY`. Then Run All.

In [ ]:
import sys, os, glob, subprocess
REPO_URL = "https://github.com/g25ait2149/aegis.git"
DEST = "/kaggle/working/aegis_src"
if os.path.isdir(os.path.join(DEST, ".git")):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, DEST], check=False)
hits = glob.glob(DEST + "/**/aegis/__init__.py", recursive=True)
root = os.path.dirname(os.path.dirname(hits[0])) if hits else DEST
sys.path.insert(0, root)
for m in [m for m in sys.modules if m == "aegis" or m.startswith(("aegis.", "eval"))]:
    del sys.modules[m]
print("aegis repo at:", root)

In [ ]:
# jailbreakbench provides the PAIR artifacts; uninstall torchao (Kaggle ships an incompatible
# version) BEFORE transformers imports so the L2 guards load cleanly.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jailbreakbench"], check=False)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
print("deps ready")

## Secrets
Add as Kaggle Secrets: **`HF_TOKEN`** (corpus + XSTest + guards) and **`WANDB_API_KEY`** (logging).

In [ ]:
from eval.semantic_eval import run_semantic_eval
rows = run_semantic_eval(
    use_guard=True,       # tuned L2 jailbreak guard + Aegis(Fast+Guard) cascade
    qwen3guard=True,      # Qwen3Guard-0.6B content guard - expected to carry the semantic axis
    wandb_log=True,
)
rows

## What to read

- **semantic_detection** = fraction of PAIR jailbreaks flagged. Expect the **L1** detectors (Aegis-Fast, RJD-v2) to be **LOW** - PAIR is fluent persuasion with no surface tell.
- **Qwen3Guard** (content guard) should be **higher** - it reads harmful *intent* regardless of framing.
- **benign_fpr** = over-refusal control (lower is better).

The gap is the answer to the professor's question: semantic attacks are an **L2** problem, not an L1 one - the same layered division of labour the obfuscation and harmful-topic results already showed, now on the semantic axis.